# Predicting Electric Vehicle Purchases
2026 Kaggle Playground Series - Season 6 Episode 9

## Setup

In [ ]:
import os
import warnings
from pathlib import Path
from types import SimpleNamespace

import kagglehub
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from scipy import stats
from scipy.special import expit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import TargetEncoder

warnings.simplefilter("ignore")

In [ ]:
COMP = "playground-series-s6e9"
LOCAL = Path("data")
DATA_DIR = LOCAL if LOCAL.exists() else Path(kagglehub.competition_download(COMP))

TARGET = "Will_Buy_EV"
ID_COL = "id"

N_FOLDS = 10
SEED = 42

## Competition Data

In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv", index_col=ID_COL)
test = pd.read_csv(DATA_DIR / "test.csv", index_col=ID_COL)

train[TARGET] = train[TARGET].map({"Yes": 1, "No": 0}).astype("int8")

X, y = train.drop(columns=[TARGET]), train[TARGET]
X_test = test.copy()

## Feature Engineering

In [ ]:
def preprocessing(df: pd.DataFrame) -> pd.DataFrame:
    # City_Type allein bringt nichts - als Teil eines Target-Encoding-Schluessels zusammen mit den ganzen Pendel-km aber schon.
    # Deshalb den Schluessel hier bilden, bevor die Spalte entfernt wird.
    city_code = df["City_Type"].map({"Suburban": 0, "Rural": 1, "Urban": 2}).astype("int64")
    df["city_commute_integer"] = city_code * 1000 + np.floor(df["Daily_Commute_km"]).astype("int64")

    # Im Feature-Test ohne Nutzen gemessen - deshalb gar nicht erst kodieren
    df = df.drop(columns=["Gender", "City_Type"])

    df["Current_Car_Type"] = df["Current_Car_Type"].map({"Sedan": 0, "SUV": 1, "Hatchback": 2, "Truck": 3}).astype("int8")

    for col in ["Home_Charging_Possible", "Subsidy_Available"]:
        df[col] = df[col].map({"Yes": 1, "No": 0}).astype("int8")

    df["Range_Anxiety_Level"] = df["Range_Anxiety_Level"].map({"Low": 0, "Medium": 1, "High": 2}).astype("int8")

    return df


X, X_test = preprocessing(X), preprocessing(X_test)

In [ ]:
def add_features(df: pd.DataFrame, income_counts: pd.Series) -> pd.DataFrame:
    df = df.copy()

    # Wie oft genau dieses Einkommen vorkommt.
    df["income_counts"] = df["Annual_Income_USD"].map(income_counts)

    # Nachkommastelle der Pendelstrecke: erst in ganze Zehntel-km umrechnen, dann die letzte Ziffer nehmen.
    # round ist ein Sicherheitsnetz gegen Float-Ungenauigkeit
    commute_tenths = np.round(df["Daily_Commute_km"] * 10).astype("int64")
    df["commute_decimal"] = (commute_tenths % 10).astype("int8")

    # Einkommen auf 10, 100 und 1000 USD abgerundet: seltene Einzelwerte werden zu groesseren Gruppen.
    for step in [10, 100, 1000]:
        df[f"income{step}_floor"] = (df["Annual_Income_USD"] // step).astype("int64")

    df["commute_integer"] = np.floor(df["Daily_Commute_km"]).astype("int64")

    return df


income_counts = pd.concat([X["Annual_Income_USD"], X_test["Annual_Income_USD"]]).value_counts()

X, X_test = add_features(X, income_counts), add_features(X_test, income_counts)
X.head(3)

## Target Encoding

In [ ]:
HELPER_COLS = [
    "income10_floor",
    "income100_floor",
    "income1000_floor",
    "commute_integer",
    "city_commute_integer"
]
# Die Hilfsspalten werden nur fuer das TE gebraucht und danach wieder entfernt
TE_COLS = ["Age", "Annual_Income_USD", "Daily_Commute_km"] + HELPER_COLS
TE_NAMES = [f"{c}_te" for c in TE_COLS]

EXTRA_SMOOTH = 100
EXTRA_SMOOTH_COLS = ["Annual_Income_USD"]
EXTRA_TE_NAMES = [f"{c}_te{EXTRA_SMOOTH}" for c in EXTRA_SMOOTH_COLS]


def add_target_encoding(X_tr: pd.DataFrame, y_tr: pd.Series, X_va: pd.DataFrame, X_te: pd.DataFrame, seed: int):
    X_tr, X_va, X_te = X_tr.copy(), X_va.copy(), X_te.copy()

    te = TargetEncoder(target_type="binary", smooth=1.0, cv=5, random_state=seed)
    X_tr[TE_NAMES] = te.fit_transform(X_tr[TE_COLS], y_tr)
    X_va[TE_NAMES] = te.transform(X_va[TE_COLS])
    X_te[TE_NAMES] = te.transform(X_te[TE_COLS])

    te_extra = TargetEncoder(target_type="binary", smooth=EXTRA_SMOOTH, cv=5, random_state=seed)
    X_tr[EXTRA_TE_NAMES] = te_extra.fit_transform(X_tr[EXTRA_SMOOTH_COLS], y_tr)
    X_va[EXTRA_TE_NAMES] = te_extra.transform(X_va[EXTRA_SMOOTH_COLS])
    X_te[EXTRA_TE_NAMES] = te_extra.transform(X_te[EXTRA_SMOOTH_COLS])

    return X_tr.drop(columns=HELPER_COLS), X_va.drop(columns=HELPER_COLS), X_te.drop(columns=HELPER_COLS)

## Model Configuration

In [ ]:
params = dict(
    num_iterations=10_000,
    learning_rate=0.05,
    num_leaves=6,
    num_threads=os.cpu_count(),
    deterministic=True,
    force_row_wise=True,
    max_depth=3,
    # Stochastic Gradient Boosting: pro Baum eine neue 80-%-Stichprobe der Zeilen.
    # Die Baeume werden dadurch unterschiedlicher und ihr Rauschen mittelt sich teilweise heraus.
    subsample=0.8,
    subsample_freq=1,
    # Stueckweise lineare Blaetter statt konstanter.
    # Das Target ist ein Probit einer linearen Punktzahl und ein Baum kann eine Gerade nur treppenfoermig nachbauen - genau diese Luecke schliesst linear_tree.
    linear_tree=True,
    linear_lambda=0.1,
    verbose=-1,
    metric="auc",
    early_stopping_rounds=500
)

## Cross-Validation

In [ ]:
def recipe_score(df: pd.DataFrame):
    return (
        1.2 * df.Annual_Income_USD / 1e5
        + 0.6 * df.Environmental_Concern_Level
        + 2.0 * df.Subsidy_Available
        - 1.0 * (df.Range_Anxiety_Level == 1)
        - 3.0 * (df.Range_Anxiety_Level == 2)
    ).values


def recipe_margins(X_tr, y_tr, X_others):
    # Nur auf dem Train-Fold gefittet; zwei Parameter auf 600k Zeilen, kein Overfitting-Thema
    calibrator = LogisticRegression(C=1e4, max_iter=1000)
    calibrator.fit(recipe_score(X_tr).reshape(-1, 1), y_tr)
    margins = [calibrator.decision_function(recipe_score(X_tr).reshape(-1, 1))]
    for X_o in X_others:
        margins.append(calibrator.decision_function(recipe_score(X_o).reshape(-1, 1)))
    return margins


def predict_probability(model, X_part, margin):
    # raw_score enthaelt nur die Baeume - die Startvorhersage selbst muessen wir dazurechnen
    return expit(margin + model.predict(X_part, raw_score=True))

In [ ]:
def run_cross_validation(params: dict, seed: int = SEED, verbose: bool = False, drop_columns: list[str] | None = None) -> SimpleNamespace:
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)

    oof_pred, test_pred = np.zeros(len(X)), np.zeros(len(X_test))
    fold_scores = []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), start=1):
        X_tr, y_tr = X.iloc[tr_idx].copy(), y.iloc[tr_idx]
        X_va, y_va = X.iloc[va_idx].copy(), y.iloc[va_idx]

        m_tr, m_va, m_te = recipe_margins(X_tr, y_tr, [X_va, X_test])
        X_tr, X_va, X_te = add_target_encoding(X_tr, y_tr, X_va, X_test, seed)

        if drop_columns is not None:
            X_tr, X_va, X_te = (d.drop(columns=drop_columns) for d in (X_tr, X_va, X_te))

        gbm = LGBMClassifier(**params, random_state=seed)
        gbm.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], init_score=m_tr, eval_init_score=[m_va])

        oof_pred[va_idx] = predict_probability(gbm, X_va, m_va)
        fold_scores.append(roc_auc_score(y_va, oof_pred[va_idx]))
        test_pred += predict_probability(gbm, X_te, m_te) / N_FOLDS

        if verbose:
            print(f"Fold {fold}: OOF-AUC {fold_scores[-1]:.6f} | bestIteration = {gbm.best_iteration_}")

    return SimpleNamespace(oof_pred=oof_pred, test_pred=test_pred, fold_scores=fold_scores, oof_auc=roc_auc_score(y, oof_pred))


def report(result: SimpleNamespace, name: str = "") -> None:
    print(f"{name}    OOF-AUC = {result.oof_auc:.6f}    mean = {np.mean(result.fold_scores):.6f}    std_dev = {np.std(result.fold_scores):.6f}")


base = run_cross_validation(params, verbose=True)
report(base, "Submission Run")

## Feature Testing

In [ ]:
def fold_delta_test(auc_with, auc_without):
    """t-Test auf die Fold-Deltas (mit - ohne), korrigiert nach Nadeau & Bengio (2003).

    Die Trainingsmengen der Folds ueberlappen (bei 10 Folds zu 90 %), die Deltas sind also nicht unabhaengig.
    Deshalb SD * sqrt(1/n + n_test/n_train) statt SD / sqrt(n) - bei 10 Folds ein 1.45-mal groesserer Standardfehler.
    """
    delta = np.asarray(auc_with) - np.asarray(auc_without)  # positiv = das Feature hilft
    n = len(delta)
    standard_error = delta.std(ddof=1) * np.sqrt(1 / n + 1 / (n - 1))  # n_test/n_train = 1/(n-1)
    t_value = delta.mean() / standard_error
    p_value = 2 * stats.t.sf(abs(t_value), df=n - 1)
    t_critical = stats.t.ppf(0.975, df=n - 1)

    return SimpleNamespace(delta=delta, mean=delta.mean(), sd=delta.std(ddof=1), t=t_value, t_critical=t_critical, p=p_value,
                           ci=(delta.mean() - t_critical * standard_error, delta.mean() + t_critical * standard_error))


def test_feature(columns, seed: int = SEED):
    """Vergleicht das Modell mit und ohne 'columns' auf identischen Folds - ein Seed, t-Test auf die Fold-Deltas.

    'columns' ist ein Spaltenname oder eine Liste. Eine Liste wird als Gruppe GEMEINSAM entfernt -
    das ist nicht dasselbe wie die Summe der Einzeltests, weil Spalten sich vertreten oder ergaenzen koennen.
    """
    if isinstance(columns, str):
        columns = [columns]

    with_feature = run_cross_validation(params, seed=seed)
    without_feature = run_cross_validation(params, seed=seed, drop_columns=columns)
    result = fold_delta_test(with_feature.fold_scores, without_feature.fold_scores)
    result.oof_difference = with_feature.oof_auc - without_feature.oof_auc

    per_fold = "  ".join(f"{value:+.6f}" for value in result.delta)

    print(f"feature:          {', '.join(columns)}")
    print(f"auc with:         {with_feature.oof_auc:.6f}")
    print(f"auc without:      {without_feature.oof_auc:.6f}")
    print("--------")
    print(f"difference:       {result.oof_difference:+.6f}  (OOF)")
    print()
    print(f"per fold:         {per_fold}")
    print(f"mean +/- SD:      {result.mean:+.6f} +/- {result.sd:.6f}")
    print(f"t (corrected):    {result.t:.1f}   (critical {result.t_critical:.2f}, p = {result.p:.3f})")
    print(f"95% CI:           {result.ci[0]:+.6f} bis {result.ci[1]:+.6f}")
    print()


test_feature(["commute_integer_te", "city_commute_integer_te"])

## Multi-Seed Ensemble

In [ ]:
runs = []
for s in range(SEED, SEED + 20):
    result = run_cross_validation(params, seed=s)
    report(result, f"Seed {s}")
    runs.append(result)

oof_pred = np.mean([r.oof_pred for r in runs], axis=0)
test_pred = np.mean([r.test_pred for r in runs], axis=0)
print(f"\nSeed-Average ueber {len(runs)} Seed(s): OOF-AUC={roc_auc_score(y, oof_pred):.6f}")

## OOF and Test Predictions

In [ ]:
np.save("oof_lgbm_base_margin.npy", oof_pred)
np.save("test_lgbm_base_margin.npy", test_pred)

submission = pd.DataFrame({TARGET: test_pred}, index=X_test.index)
submission.to_csv("submission.csv")
submission.head()